# Meteorite Baseline Training

这个 notebook 是当前项目的主训练入口。

设计原则：
- notebook 负责改参数、启动训练、看结果
- `models/` 和 `tools/` 负责真正的底层逻辑
- 这样既适合学习，也适合后续在服务器上长期维护

## 1. 导入依赖

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/data/数据科学实践project").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)
print((PROJECT_ROOT / "models").exists())
print((PROJECT_ROOT / "tools").exists())


In [ ]:
from pathlib import Path
import json
import pandas as pd


import torch
import torch.nn as nn
from torch.utils.data import DataLoader

#near_deplicate审查
import numpy as np
from PIL import Image, ImageOps
from tqdm.auto import tqdm


from models import build_model
from tools.dataset_tools import (
    MeteoriteDataset,
    attach_md5_group_ids,
    group_aware_train_val_split,
    load_test_dataframe,
    load_train_dataframe,
    stratified_sample_dataframe,
    stratified_train_val_split,
)


from tools.submission_tools import make_submission, predict_on_test
from tools.train_tools import (
    IntegerLabelBCEWithLogitsLoss,
    build_bce_pos_weight,
    evaluate,
    set_seed,
    train_one_epoch,
)
from tools.transforms_tools import build_eval_transforms, build_train_transforms
from tools.visualize_tools import plot_label_distribution, plot_training_curves, save_sample_grid


## 2. 配置训练参数

这里是你最常改的地方。

这版会自动把重要参数编码进实验目录名，并把完整参数保存到 `run_summary.json`。

`loss_name` 可选 `cross_entropy` 或 `bce_with_logits`。

In [ ]:
from datetime import datetime

CONFIG = {
    'data_root': PROJECT_ROOT,
    'output_base_dir': PROJECT_ROOT / 'runs',

    'model_name': 'resnet18',
    'pretrained': True,

    'epochs': 20,
    'batch_size': 16,
    'img_size': 224,
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'val_ratio': 0.2,
    'scheduler': 'cosine',
    'loss_name': 'cross_entropy',  # 可选: 'cross_entropy', 'bce_with_logits'

    'subset_size': 0,
    'test_subset_size': 0,

    'seed': 42,
    'num_workers': 4,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',

    # TTA 参数
    'tta_repeats': 8,  # 每张测试图推理次数，越多越稳但越慢
}



def format_float_for_name(value):
    text = f"{value:g}"
    return text.replace("+", "")


base_exp_name = '_'.join([
    f"img{CONFIG['img_size']}",
    CONFIG['model_name'],
    'adamw',
    f"lr{format_float_for_name(CONFIG['lr'])}",
    f"wd{format_float_for_name(CONFIG['weight_decay'])}",
    f"bs{CONFIG['batch_size']}",
    f"val{format_float_for_name(CONFIG['val_ratio'])}",
    f"seed{CONFIG['seed']}",
    f"sch{CONFIG['scheduler']}",
    f"loss{CONFIG['loss_name']}",
    f"pre{int(bool(CONFIG['pretrained']))}",
    f"subset{CONFIG['subset_size'] if CONFIG['subset_size'] and CONFIG['subset_size'] > 0 else 'full'}",
    f"e{CONFIG['epochs']}",
])

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

CONFIG['experiment_name'] = f"{timestamp}_{base_exp_name}"
CONFIG['output_dir'] = CONFIG['output_base_dir'] / CONFIG['experiment_name']
CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)

print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()}, ensure_ascii=False, indent=2))


## 3. 固定随机种子并读取数据

In [ ]:
def count_cross_split_groups(df_train, df_val, group_col):
    """
    统计某个分组列在 train / val 同时出现的组数量。
    理论上如果按这个 group_col 做 group-aware split，结果应该是 0。
    """
    if group_col is None:
        return None
    if group_col not in df_train.columns or group_col not in df_val.columns:
        return None

    train_groups = set(df_train[group_col].dropna().astype(str).tolist())
    val_groups = set(df_val[group_col].dropna().astype(str).tolist())
    return len(train_groups.intersection(val_groups))


def drop_exact_duplicates_in_train(df, group_col='group_id'):
    """
    训练集内部 exact duplicate 去重：
    同一个 md5 group 只保留 1 张。
    """
    if group_col not in df.columns:
        raise ValueError(f'列 {group_col} 不存在，无法去重。')

    candidate_path_cols = ['img_path', 'image_path', 'filepath', 'file_path', 'path']
    sort_cols = [col for col in candidate_path_cols if col in df.columns]

    if sort_cols:
        dedup_df = (
            df.sort_values([group_col] + sort_cols)
              .drop_duplicates(subset=group_col, keep='first')
              .reset_index(drop=True)
        )
    else:
        dedup_df = (
            df.sort_values(group_col)
              .drop_duplicates(subset=group_col, keep='first')
              .reset_index(drop=True)
        )

    removed_count = len(df) - len(dedup_df)
    return dedup_df, removed_count


def compute_dhash_int(image_path, hash_size=8):
    """
    计算单张图片的 dHash，返回一个整数。
    """
    image = Image.open(image_path)
    image = ImageOps.exif_transpose(image).convert('L')
    image = image.resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)

    pixels = np.asarray(image, dtype=np.int16)
    diff = pixels[:, 1:] > pixels[:, :-1]

    hash_value = 0
    for bit in diff.flatten():
        hash_value = (hash_value << 1) | int(bit)

    return hash_value


def hamming_distance_int(x, y):
    """
    计算两个整数哈希的汉明距离。
    pandas 在含缺失值的列里可能把整数哈希提升成 float，
    这里显式转成 Python int，避免位运算类型报错。
    """
    return (int(x) ^ int(y)).bit_count()


class UnionFind:
    """
    并查集，用来把彼此相似的 md5 group 合并成一个 near-duplicate family。
    """
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra = self.find(a)
        rb = self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            self.parent[ra] = rb
        elif self.rank[ra] > self.rank[rb]:
            self.parent[rb] = ra
        else:
            self.parent[rb] = ra
            self.rank[ra] += 1


def build_near_duplicate_families_from_md5_groups(
    df,
    path_col='img_path',
    md5_group_col='group_id',
    threshold=6,
    hash_size=8,
):
    """
    先按 md5 group 选代表图，再用 dHash 把高相似 group 合并成 near-duplicate family。
    """
    if md5_group_col not in df.columns:
        raise ValueError(f'列 {md5_group_col} 不存在，无法构建 near-duplicate family。')
    if path_col not in df.columns:
        raise ValueError(f'列 {path_col} 不存在，无法读取图片。')

    rep_df = (
        df.sort_values([md5_group_col, path_col])
          .drop_duplicates(subset=md5_group_col, keep='first')
          .reset_index(drop=True)
          .copy()
    )

    dhash_list = []
    failed_paths = []

    for image_path in tqdm(rep_df[path_col].tolist(), desc='计算 md5-group 代表图 dHash'):
        try:
            dhash_list.append(compute_dhash_int(image_path, hash_size=hash_size))
        except Exception:
            dhash_list.append(None)
            failed_paths.append(str(image_path))

    # dHash 可能达到 64 位无符号整数上限，不能用 pandas 的有符号 Int64 保存。
    # 这里保持为 object 列，元素是 Python int / None，避免溢出。
    rep_df['dhash'] = pd.Series(dhash_list, dtype='object')

    valid_mask = rep_df['dhash'].notna()
    valid_rep_df = rep_df[valid_mask].reset_index(drop=True).copy()
    valid_rep_df['dhash'] = valid_rep_df['dhash'].map(int)

    n = len(valid_rep_df)
    uf = UnionFind(n)

    for i in tqdm(range(n), desc='构建 near-duplicate family'):
        hi = valid_rep_df.loc[i, 'dhash']
        for j in range(i + 1, n):
            hj = valid_rep_df.loc[j, 'dhash']
            if hamming_distance_int(hi, hj) <= threshold:
                uf.union(i, j)

    root_to_family = {}
    family_ids = []

    for i in range(n):
        root = uf.find(i)
        if root not in root_to_family:
            root_to_family[root] = f'nearfam_{len(root_to_family):05d}'
        family_ids.append(root_to_family[root])

    valid_rep_df['near_dup_family_id'] = family_ids

    invalid_rep_df = rep_df[~valid_mask].copy()
    invalid_rep_df['near_dup_family_id'] = [
        f'nearfam_failed_{i:05d}' for i in range(len(invalid_rep_df))
    ]

    rep_family_df = pd.concat(
        [
            valid_rep_df[[md5_group_col, 'dhash', 'near_dup_family_id']],
            invalid_rep_df[[md5_group_col, 'dhash', 'near_dup_family_id']],
        ],
        axis=0,
        ignore_index=True,
    )


    out_df = df.merge(
        rep_family_df[[md5_group_col, 'near_dup_family_id', 'dhash']],
        on=md5_group_col,
        how='left',
    )

    return out_df, rep_family_df, failed_paths


In [ ]:
set_seed(CONFIG['seed'])
device = torch.device(CONFIG['device'])
print('device =', device)

full_train_df = load_train_dataframe(CONFIG['data_root'])
print('完整训练样本数 =', len(full_train_df))

# ============================================================
# 补充数据集：将 Dataset/ 下所有图片加入训练集
# - 除了 Meteorite 文件夹标记为 label=1（陨石/正类）
# - 其余文件夹标记为 label=0（岩石/负类）
# 兼容两种目录结构：
#   1) 嵌套：Folder/Subfolder/images （如 Igneous/Basalt/xxx.jpg）
#   2) 扁平：Folder/images           （如 Amphibolite/xxx.jpg）
# ============================================================
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tif', '.tiff'}
dataset_dir = CONFIG['data_root'] / "Dataset"
rock_rows = []

for rock_type_dir in sorted(dataset_dir.iterdir()):
    if not rock_type_dir.is_dir() or rock_type_dir.name.startswith('.'):
        continue
    # Meteorite 文件夹是正类样本，后面单独处理
    if rock_type_dir.name == 'Meteorite':
        continue

    sub_items = list(rock_type_dir.iterdir())
    has_subdirs = any(p.is_dir() and not p.name.startswith('.') for p in sub_items)
    has_images  = any(p.is_file() and p.suffix.lower() in VALID_EXTENSIONS for p in sub_items)

    if has_subdirs:
        for subtype_dir in sorted(rock_type_dir.iterdir()):
            if not subtype_dir.is_dir() or subtype_dir.name.startswith('.'):
                continue
            for img_path in subtype_dir.glob("*"):
                if img_path.suffix.lower() in VALID_EXTENSIONS:
                    unique_id = f"{rock_type_dir.name}_{subtype_dir.name}_{img_path.name}"
                    rock_rows.append({'id': unique_id, 'label': 0, 'img_path': str(img_path)})
    elif has_images:
        for img_path in rock_type_dir.glob("*"):
            if img_path.is_file() and img_path.suffix.lower() in VALID_EXTENSIONS:
                unique_id = f"{rock_type_dir.name}_{img_path.name}"
                rock_rows.append({'id': unique_id, 'label': 0, 'img_path': str(img_path)})

rock_df = pd.DataFrame(rock_rows).sample(frac=1.0, random_state=CONFIG['seed']).reset_index(drop=True)
print(f"补充岩石图片总数: {len(rock_df)}")

# ============================================================
# 控制负样本比例：保持 label=0 与 label=1 的比例不超过 target_ratio
# target_ratio = 1.0  →  1:1 平衡
# target_ratio = 1.5  → 负类最多是正类的 1.5 倍
# target_ratio = None → 全部使用（不做限制）
# ============================================================
target_ratio = 1.5   # 改为 None 则不加限制

n_positive = int(full_train_df['label'].sum())  # label=1 的数量

if target_ratio is not None:
    max_negatives = int(n_positive * target_ratio)
    n_original_neg = int((full_train_df['label'] == 0).sum())
    # 计算最多能从补充数据中取多少负样本
    max_supplement = max(0, max_negatives - n_original_neg)
    if len(rock_df) > max_supplement:
        rock_df = rock_df.iloc[:max_supplement]
        print(f"补充岩石数据已截断: {max_supplement} 张 (保持负:正 ≤ {target_ratio}:1)")
else:
    print("未限制负样本数量（全部使用）")

print(f"实际加入补充岩石图片: {len(rock_df)}")

# ============================================================
# 补充陨石正类样本：读取 Dataset/Meteorite/ 下的图片，标记为 label=1
# ============================================================
meteorite_dir = dataset_dir / "Meteorite"
meteorite_rows = []

if meteorite_dir.is_dir():
    sub_items = list(meteorite_dir.iterdir())
    has_subdirs = any(p.is_dir() and not p.name.startswith('.') for p in sub_items)
    has_images  = any(p.is_file() and p.suffix.lower() in VALID_EXTENSIONS for p in sub_items)

    if has_subdirs:
        for subtype_dir in sorted(meteorite_dir.iterdir()):
            if not subtype_dir.is_dir() or subtype_dir.name.startswith('.'):
                continue
            for img_path in subtype_dir.glob("*"):
                if img_path.suffix.lower() in VALID_EXTENSIONS:
                    unique_id = f"Meteorite_{subtype_dir.name}_{img_path.name}"
                    meteorite_rows.append({'id': unique_id, 'label': 1, 'img_path': str(img_path)})
    elif has_images:
        for img_path in meteorite_dir.glob("*"):
            if img_path.is_file() and img_path.suffix.lower() in VALID_EXTENSIONS:
                unique_id = f"Meteorite_{img_path.name}"
                meteorite_rows.append({'id': unique_id, 'label': 1, 'img_path': str(img_path)})

meteorite_df = pd.DataFrame(meteorite_rows).sample(frac=1.0, random_state=CONFIG['seed']).reset_index(drop=True)
print(f"补充陨石图片总数: {len(meteorite_df)}")
# 正类样本珍贵，不做数量限制，全部加入

# 与原始训练集合并，并随机打乱
full_train_df = pd.concat([full_train_df, rock_df, meteorite_df], ignore_index=True)
full_train_df = full_train_df.sample(frac=1.0, random_state=CONFIG['seed']).reset_index(drop=True)
print(f"合并后训练样本总数: {len(full_train_df)}")
print(f"合并后类别分布:\n{full_train_df['label'].value_counts().sort_index()}")

train_pool_df = stratified_sample_dataframe(
    df=full_train_df,
    label_col='label',
    sample_size=CONFIG['subset_size'],
    seed=CONFIG['seed'],
)
print('抽样后训练池样本数 =', len(train_pool_df))

# 第一步：先做 md5 exact duplicate 分组
train_pool_df = attach_md5_group_ids(train_pool_df)
md5_group_col = 'group_id'
path_col = 'img_path'

print('唯一 md5 group 数 =', train_pool_df[md5_group_col].nunique())

# 第二步：在 md5 group 基础上构建 near-duplicate family
near_dup_threshold = 6
near_dup_hash_method = 'dhash'

train_pool_df, near_dup_rep_df, near_dup_failed_paths = build_near_duplicate_families_from_md5_groups(
    df=train_pool_df,
    path_col=path_col,
    md5_group_col=md5_group_col,
    threshold=near_dup_threshold,
    hash_size=8,
)

print('near-duplicate family 数 =', train_pool_df['near_dup_family_id'].nunique())
print('dHash 失败图片数 =', len(near_dup_failed_paths))

# 第三步：按 near-duplicate family 做 group-aware split
split_method = 'group_aware_neardup_dhash'
group_key = 'near_dup_family_id'

train_df, val_df = group_aware_train_val_split(
    df=train_pool_df,
    label_col='label',
    group_col=group_key,
    val_ratio=CONFIG['val_ratio'],
    seed=CONFIG['seed'],
)
print('切分后: train =', len(train_df), 'val =', len(val_df))

# 第四步：做切分泄漏审计
cross_split_dup_count = count_cross_split_groups(train_df, val_df, md5_group_col)
cross_split_neardup_count = count_cross_split_groups(train_df, val_df, group_key)

print('split_method =', split_method)
print('group_key =', group_key)
print('cross_split_dup_count =', cross_split_dup_count)
print('cross_split_neardup_count =', cross_split_neardup_count)

# 第五步：训练集内部 exact duplicate 去重
train_size_before_exact_dedup = len(train_df)
train_df, train_exact_dup_removed = drop_exact_duplicates_in_train(
    train_df,
    group_col=md5_group_col,
)
train_size_after_exact_dedup = len(train_df)

print('训练集 exact duplicate 去重前 =', train_size_before_exact_dedup)
print('训练集 exact duplicate 去重后 =', train_size_after_exact_dedup)
print('训练集移除的重复样本数 =', train_exact_dup_removed)

if cross_split_dup_count not in (None, 0):
    print(f'[WARN] md5 重复组仍跨 train/val: {cross_split_dup_count}')

if cross_split_neardup_count not in (None, 0):
    print(f'[WARN] near-duplicate family 仍跨 train/val: {cross_split_neardup_count}')


## 4. 保存切分信息和基础可视化

In [ ]:
split_dir = CONFIG['output_dir'] / 'splits'
fig_dir = CONFIG['output_dir'] / 'figures'
split_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

train_df[['id', 'label', 'img_path']].to_csv(split_dir / 'train_split.csv', index=False)
val_df[['id', 'label', 'img_path']].to_csv(split_dir / 'val_split.csv', index=False)

plot_label_distribution(train_df, fig_dir / 'train_label_distribution.png')
plot_label_distribution(val_df, fig_dir / 'val_label_distribution.png')
print('已保存类别分布图')


## 5. 构建 Dataset 和 DataLoader

In [ ]:
train_tfms = build_train_transforms(img_size=CONFIG['img_size'])
eval_tfms = build_eval_transforms(img_size=CONFIG['img_size'])

train_dataset = MeteoriteDataset(train_df, transforms=train_tfms)
val_dataset = MeteoriteDataset(val_df, transforms=eval_tfms)

save_sample_grid(train_dataset, fig_dir / 'train_samples.png', max_images=16)
print('已保存样本网格图')

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    pin_memory=(device.type == 'cuda'),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=(device.type == 'cuda'),
)


## 6. 构建模型、损失函数、优化器和学习率调度器

In [ ]:
model = build_model(
    name=CONFIG['model_name'],
    num_classes=2,
    pretrained=CONFIG['pretrained'],
).to(device)

class_counts = train_df['label'].value_counts().sort_index()
total_count = int(class_counts.sum())
class_weights = torch.tensor(
    [
        total_count / max(int(class_counts.get(0, 1)), 1),
        total_count / max(int(class_counts.get(1, 1)), 1),
    ],
    dtype=torch.float32,
    device=device,
)
if CONFIG['loss_name'] == 'cross_entropy':
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
elif CONFIG['loss_name'] == 'bce_with_logits':
    pos_weight = build_bce_pos_weight(class_counts.to_list(), device=device)
    criterion = IntegerLabelBCEWithLogitsLoss(pos_weight=pos_weight)
else:
    raise ValueError("CONFIG['loss_name'] 只能是 'cross_entropy' 或 'bce_with_logits'")
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])

if CONFIG['scheduler'] == 'cosine':
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer=optimizer,
        T_max=CONFIG['epochs'],
        eta_min=1e-6,
    )
else:
    scheduler = None

print(model.__class__.__name__)


## 7. 开始训练

In [ ]:
history = []
best_val_f1 = -1.0
best_ckpt_path = CONFIG['output_dir'] / 'best_model.pt'

for epoch in range(1, CONFIG['epochs'] + 1):
    print(f'\n========== Epoch {epoch}/{CONFIG["epochs"]} ==========')
    current_lr = optimizer.param_groups[0]['lr']

    train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
    )
    val_metrics = evaluate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
    )

    row = {
        'epoch': epoch,
        'train_loss': train_metrics['loss'],
        'train_acc': train_metrics['acc'],
        'train_f1': train_metrics['f1'],
        'val_loss': val_metrics['loss'],
        'val_acc': val_metrics['acc'],
        'val_f1': val_metrics['f1'],
        'lr': current_lr,
    }
    history.append(row)

    print(
        f"train_loss={row['train_loss']:.4f}, train_acc={row['train_acc']:.4f}, train_f1={row['train_f1']:.4f} | "
        f"val_loss={row['val_loss']:.4f}, val_acc={row['val_acc']:.4f}, val_f1={row['val_f1']:.4f} | "
        f"lr={row['lr']:.6f}"
    )

    if row['val_f1'] > best_val_f1:
        best_val_f1 = row['val_f1']
        torch.save(
            {
                'model_state_dict': model.state_dict(),
                'epoch': epoch,
                'val_f1': best_val_f1,
                'config': {k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()},
                'experiment_name': CONFIG['experiment_name'],
            },
            best_ckpt_path,
        )
        print('已保存更优模型 ->', best_ckpt_path)

    if scheduler is not None:
        scheduler.step()


## 8. 保存训练历史、训练曲线和运行摘要

In [ ]:
metrics_path = CONFIG['output_dir'] / 'metrics_history.json'
metrics_path.write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding='utf-8')
plot_training_curves(history, CONFIG['output_dir'] / 'figures' / 'curves.png')

run_summary = {
    'experiment_name': CONFIG['experiment_name'],
    'output_dir': str(CONFIG['output_dir']),
    'data_root': str(CONFIG['data_root']),
    'best_val_f1': best_val_f1,
    'epochs': CONFIG['epochs'],
    'train_size': len(train_df),
    'val_size': len(val_df),
    'model_name': CONFIG['model_name'],
    'pretrained': CONFIG['pretrained'],
    'loss_name': CONFIG['loss_name'],
    'scheduler': CONFIG['scheduler'],
    'device': str(device),

    # split 审计
    'split_method': split_method,
    'group_key': group_key,
    'cross_split_dup_count': cross_split_dup_count,
    'cross_split_neardup_count': cross_split_neardup_count,

    # near-duplicate 审计信息
    'near_dup_hash_method': near_dup_hash_method,
    'near_dup_threshold': near_dup_threshold,
    'near_dup_family_count': int(train_pool_df['near_dup_family_id'].nunique()),
    'near_dup_rep_count': int(len(near_dup_rep_df)),
    'near_dup_failed_count': int(len(near_dup_failed_paths)),

    # 训练集 exact dedup 审计
    'train_size_before_exact_dedup': train_size_before_exact_dedup,
    'train_size_after_exact_dedup': train_size_after_exact_dedup,
    'train_exact_dup_removed': train_exact_dup_removed,

    'all_config': {k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()},
    'class_counts_train_split': {str(k): int(v) for k, v in train_df['label'].value_counts().sort_index().items()},
    'class_counts_val_split': {str(k): int(v) for k, v in val_df['label'].value_counts().sort_index().items()},
}

(CONFIG['output_dir'] / 'run_summary.json').write_text(
    json.dumps(run_summary, ensure_ascii=False, indent=2),
    encoding='utf-8'
)

# 顺手把 near-duplicate family 代表表也存下来，方便你后面人工抽查
near_dup_rep_path = CONFIG['output_dir'] / 'near_dup_representatives.csv'
near_dup_rep_df.to_csv(near_dup_rep_path, index=False, encoding='utf-8-sig')

print('已保存 metrics_history.json、run_summary.json、curves.png 和 near_dup_representatives.csv')


## 9. 测试集推理与提交文件生成

In [ ]:
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm


@torch.no_grad()
def predict_test_probabilities(model, loader, device):
    """
    在测试集上推理，返回一个 DataFrame:
    - id: 图片文件名
    - prob: 预测为正类(label=1)的概率
    - logit: 正类分数（便于调试）
    """
    model.eval()

    rows = []

    for images, img_paths in tqdm(loader, desc='Predicting probabilities', leave=False):
        images = images.to(device, non_blocking=True)
        logits = model(images)

        # 兼容两种二分类输出格式：
        # 1) [B, 2]：两个类别的 logits
        # 2) [B, 1] 或 [B]：单个 logit，用 sigmoid
        if logits.ndim == 2 and logits.shape[1] == 2:
            if CONFIG['loss_name'] == 'bce_with_logits':
                probs = torch.sigmoid(logits[:, 1])
            else:
                probs = torch.softmax(logits, dim=1)[:, 1]
            pos_logits = logits[:, 1]
        else:
            pos_logits = logits.view(-1)
            probs = torch.sigmoid(pos_logits)

        probs = probs.detach().cpu().tolist()
        pos_logits = pos_logits.detach().cpu().tolist()

        for path, prob, logit in zip(img_paths, probs, pos_logits):
            image_id = Path(path).name
            rows.append({
                'id': image_id,
                'prob': float(prob),
                'logit': float(logit),
            })

    pred_df = pd.DataFrame(rows).sort_values('prob', ascending=False).reset_index(drop=True)
    return pred_df


@torch.no_grad()
def predict_test_probabilities_tta(model, test_df, train_transforms, device, repeats=8):
    """
    TTA (Test Time Augmentation) 推理：
    对每张测试图用训练增强（含随机退化）推理 repeats 次，取平均概率。
    多次随机增强的平均值比单次干净推理更稳定，尤其适合低质量测试集。
    """
    from PIL import Image, ImageOps

    model.eval()
    rows = []

    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='TTA inference'):
        img_path = row['img_path']
        image_id = Path(img_path).name

        image = Image.open(img_path)
        image = ImageOps.exif_transpose(image).convert('RGB')

        prob_sum = 0.0

        for _ in range(repeats):
            img_aug = train_transforms(image)
            img_aug = img_aug.unsqueeze(0).to(device)

            logits = model(img_aug)
            if logits.ndim == 2 and logits.shape[1] == 2:
                prob = torch.softmax(logits, dim=1)[:, 1].item()
            else:
                prob = torch.sigmoid(logits.view(-1)).item()

            prob_sum += prob

        avg_prob = prob_sum / repeats

        rows.append({
            'id': image_id,
            'prob': float(avg_prob),
        })

    pred_df = pd.DataFrame(rows).sort_values('prob', ascending=False).reset_index(drop=True)
    return pred_df


def apply_prediction_rule(pred_df, mode='threshold', threshold=0.5, top_p=0.2, top_k=101):
    """
    根据不同规则，把概率转成 0/1 标签。

    mode 可选：
    - 'threshold': prob >= threshold 判为正类
    - 'top_p': 概率最高的前 p 比例样本判为正类，例如 top_p=0.2 表示前 20%
    - 'top_k': 概率最高的前 k 个样本判为正类，例如 top_k=101
    """
    out_df = pred_df.copy().sort_values('prob', ascending=False).reset_index(drop=True)
    n = len(out_df)

    if mode == 'threshold':
        out_df['label'] = (out_df['prob'] >= threshold).astype(int)
        rule_name = f'threshold{threshold:g}'

    elif mode == 'top_p':
        if not (0 < top_p <= 1):
            raise ValueError('top_p 必须在 (0, 1] 范围内，例如 0.2 表示前 20%')
        k = max(1, int(round(n * top_p)))
        out_df['label'] = 0
        out_df.loc[:k - 1, 'label'] = 1
        rule_name = f'topp{top_p:g}_top{k}'

    elif mode == 'top_k':
        if not (1 <= top_k <= n):
            raise ValueError(f'top_k 必须在 [1, {n}] 范围内')
        out_df['label'] = 0
        out_df.loc[:top_k - 1, 'label'] = 1
        rule_name = f'topk{top_k}'

    else:
        raise ValueError("mode 只能是 'threshold'、'top_p' 或 'top_k'")

    return out_df, rule_name


def save_submission_from_pred_df(template_csv_path, pred_df_with_label, output_path):
    """
    按 sample_submission.csv 的顺序生成提交文件。
    """
    template_csv_path = Path(template_csv_path)
    output_path = Path(output_path)

    sub_df = pd.read_csv(template_csv_path)
    sub_df['label'] = sub_df['id'].map(
        pred_df_with_label.set_index('id')['label'].to_dict()
    )

    if sub_df['label'].isna().any():
        missing_ids = sub_df.loc[sub_df['label'].isna(), 'id'].head(5).tolist()
        raise RuntimeError(f'以下测试图片没有预测结果: {missing_ids}')

    sub_df['label'] = sub_df['label'].astype(int)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sub_df.to_csv(output_path, index=False)
    return sub_df


In [ ]:
ckpt = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print('best_val_f1 =', ckpt['val_f1'])

test_df = load_test_dataframe(CONFIG['data_root'])
if CONFIG['test_subset_size'] and CONFIG['test_subset_size'] > 0 and CONFIG['test_subset_size'] < len(test_df):
    test_df = test_df.sample(n=CONFIG['test_subset_size'], random_state=CONFIG['seed']).reset_index(drop=True)
    print('测试集已抽样 =', len(test_df))
else:
    print('测试集样本数 =', len(test_df))

# TTA 推理：用训练增强（含随机退化）对每张图推理多次取平均
print(f'使用 TTA 推理 (repeats={CONFIG["tta_repeats"]})...')
pred_df = predict_test_probabilities_tta(
    model=model,
    test_df=test_df,
    train_transforms=train_tfms,
    device=device,
    repeats=CONFIG['tta_repeats'],
)
pred_df.to_csv(CONFIG['output_dir'] / 'test_pred_probs_tta.csv', index=False, encoding='utf-8-sig')
print('已保存 TTA 测试集概率 ->', CONFIG['output_dir'] / 'test_pred_probs_tta.csv')

# =========================================================
# 提交策略：Stage 2 使用 top_k=80
# =========================================================
submission_mode = 'top_k'
threshold_value = 0.5
top_p_value = 0.20
top_k_value = 80

pred_labeled_df, rule_name = apply_prediction_rule(
    pred_df,
    mode=submission_mode,
    threshold=threshold_value,
    top_p=top_p_value,
    top_k=top_k_value,
)

print('提交策略 =', submission_mode)
print('规则名称 =', rule_name)
print('预测正类数量 =', int(pred_labeled_df['label'].sum()))
print('预测负类数量 =', int((pred_labeled_df['label'] == 0).sum()))

# 保存带概率和标签的完整结果
pred_labeled_path = CONFIG['output_dir'] / f'test_pred_probs_{rule_name}.csv'
pred_labeled_df.to_csv(pred_labeled_path, index=False, encoding='utf-8-sig')
print('已保存带标签概率表 ->', pred_labeled_path)

# 生成提交文件
submission_path = CONFIG['output_dir'] / f'submission_{rule_name}.csv'
save_submission_from_pred_df(
    CONFIG['data_root'] / 'sample_submission.csv',
    pred_labeled_df,
    submission_path,
)
print('已生成完整提交文件 ->', submission_path)


## 10. 服务器上怎么用

推荐方式：
- 在服务器上开 JupyterLab / Jupyter Notebook，直接运行这个 notebook
- 日常调参看结果，用 notebook
- 长时间批量实验，仍然可以保留 `train_baseline.py` 做命令行运行

也就是说：
- notebook 是主入口
- `.py` 脚本是稳定后端

## 11. 命令行运行 `train_baseline.py`

如果你在服务器上想脱离 notebook 直接训练，可以使用命令行。

示例 1：快速冒烟测试
```bash
python train_baseline.py --data-root . --model simple_cnn --epochs 1 --batch-size 8 --subset-size 64 --test-subset-size 8 --img-size 224 --device cpu
```

示例 2：正式训练 ResNet18
```bash
python train_baseline.py --data-root . --model resnet18 --epochs 10 --batch-size 16 --img-size 224 --device auto
```

说明：
- 不传 `--output-dir` 时，脚本会自动生成带参数名的实验目录
- 如果你想自定义输出目录，也可以显式传入 `--output-dir runs/my_exp_name`
- 训练结束后，会自动保存 `best_model.pt`、`metrics_history.json`、`run_summary.json` 和 `figures/curves.png`